# A Code Walkthrough of the Naive Trading Engine

This notebook breaks down the implementation of `engine_naive.py`. We will look at the classes, data structures, and algorithms used to create this simple matching engine.

### 1. Core Data Structures (`Order` and `Trade`)

The entire engine is built on two fundamental data classes: `Order` and `Trade`. These classes act like simple records to hold information.

In [ ]:
import itertools
from datetime import datetime

class Order:
    """Represents a single order in the order book."""

    _ids = itertools.count(1)

    def __init__(self, side, price, quantity):
        self.id = next(self._ids) # Unique ID for the order
        self.timestamp = datetime.now() # Time the order was created
        self.side = side  # 'buy' or 'sell'
        self.price = price
        self.quantity = quantity # Original quantity
        self.remaining = quantity # Quantity left to be filled
        self.cancelled = False # Flag to mark if the order is cancelled

    def __repr__(self):
        return f"ORDER#{self.id} [{self.timestamp.strftime('%Y-%m-%d %H:%M:%S')}] {self.side.upper()}: {self.quantity} @ {self.price}"

    @property
    def is_filled(self):
        """An order is considered 'filled' if it has no remaining quantity or is cancelled."""
        return self.remaining == 0 or self.cancelled

In [ ]:
class Trade:
    """Represents a trade that has been executed."""

    _ids = itertools.count(1)

    def __init__(self, price, volume, buy_order_id, sell_order_id):
        self.id = next(self._ids)
        self.timestamp = datetime.now()
        self.price = price
        self.volume = volume
        self.buy_order_id = buy_order_id
        self.sell_order_id = sell_order_id

    def __repr__(self):
        return f"TRADE#{self.id} [{self.timestamp.strftime('%Y-%m-%d %H:%M:%S')}] {self.volume} @ {self.price}"

### 2. The `OrderBook` Class

The `OrderBook` is the main class that orchestrates everything. Its 'naive' nature comes from its choice of data structure: simple Python lists to store the buy and sell orders.

In [ ]:
from collections import defaultdict

class OrderBook:
    """A naive order book that uses simple lists and sorting for matching."""

    def __init__(self):
        # The core of the 'naive' engine: simple lists!
        self.buys = []
        self.sells = []
        
        self.trades = []
        # A dictionary to allow for fast lookups of any order by its ID.
        self.orders = {}

#### Placing an Order
When an order is placed, it's added to the master `orders` dictionary and appended to the correct list (`buys` or `sells`). Then, the matching process is immediately triggered.

In [ ]:
def place_order(self, order: Order):
    """Places an order and triggers the matching process."""
    self.orders[order.id] = order

    if order.side == "buy":
        self.buys.append(order)
    else:  # 'sell'
        self.sells.append(order)

    print(f"Placed {order}")
    self._match_orders() # Trigger matching

#### The Naive Matching Algorithm
This `_match_orders` method is the heart of the engine and also the source of its inefficiency.

On **every single trade**, it performs these steps:
1. Filters the entire `buys` and `sells` lists to remove completed orders.
2. Sorts the *entire* `buys` list by price (highest first).
3. Sorts the *entire* `sells` list by price (lowest first).
4. Enters a loop to match the best buy order with the best sell order until the prices no longer cross.

In [ ]:
def _match_orders(self):
    # Filter out any orders that are already filled or cancelled.
    self.buys = [o for o in self.buys if not o.is_filled]
    self.sells = [o for o in self.sells if not o.is_filled]

    # Sort buy orders from highest to lowest price (best for matching).
    self.buys.sort(key=lambda o: (-o.price, o.timestamp))
    # Sort sell orders from lowest to highest price (best for matching).
    self.sells.sort(key=lambda o: (o.price, o.timestamp))

    # Loop as long as there's a potential match.
    while self.buys and self.sells and self.buys[0].price >= self.sells[0].price:
        best_buy = self.buys[0]
        best_sell = self.sells[0]

        trade_quantity = min(best_buy.remaining, best_sell.remaining)
        trade_price = best_sell.price # Trade occurs at the price of the resting order

        # Create and record the trade
        trade = Trade(trade_price, trade_quantity, best_buy.id, best_sell.id)
        self.trades.append(trade)
        print(f"Executed {trade}")

        # Update the remaining quantities of the orders
        best_buy.remaining -= trade_quantity
        best_sell.remaining -= trade_quantity

        # Remove filled orders from the books by re-filtering the lists
        if best_buy.is_filled:
            self.buys.pop(0)

        if best_sell.is_filled:
            self.sells.pop(0)

### 3. Live Demonstration

Now that we've seen the implementation, let's see it in action.

In [ ]:
# We need to bring all the class and method definitions from above into scope
# In a real .py file, they would all be in the same scope already.
OrderBook._match_orders = _match_orders
OrderBook.place_order = place_order

from pprint import pprint

In [ ]:
print("--- Setting up the Order Book ---")
book = OrderBook()

print("\n--- Placing a BUY order ---")
book.place_order(Order(side="buy", price=100, quantity=10))

print("\n--- Placing a non-matching SELL order ---")
book.place_order(Order(side="sell", price=102, quantity=50))

print("\n--- Placing a matching SELL order ---")
book.place_order(Order(side="sell", price=100, quantity=10))

In [ ]:
print("--- Inspecting the final list of trades ---")
pprint(book.trades)